# Momants CSV-sentimentprocessor

Deze notebook verwerkt een Momants CSV-export met `tabularisai/multilingual-sentiment-analysis`. Er zit geen voorbeelddata in het project: vul hieronder alleen het pad naar je eigen CSV in.

De processor gebruikt `conversation_id` om berichten te groeperen en `created_at` om ze binnen ieder gesprek chronologisch te ordenen. Alleen bezoekersberichten (`from_agent == False`) worden geclassificeerd.

## 1. De processor importeren

De eigenlijke verwerking staat in `momants_sentiment.py`. Daardoor kun je dezelfde code vanuit deze notebook én vanaf de commandoregel gebruiken.

In [ ]:
from pathlib import Path
import sys

PROJECTMAP = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECTMAP) not in sys.path:
    sys.path.insert(0, str(PROJECTMAP))

from momants_sentiment import laad_momants_csv, selecteer_bezoekersberichten, verwerk_csv

## 2. Kies de CSV en uitvoermap

Vervang het voorbeeldpad door het pad naar de Momants-export. De uitvoer bevat standaard geen oorspronkelijke berichttekst.

In [ ]:
CSV_PAD = PROJECTMAP / "pad" / "naar" / "momants-export.csv"
UITVOERMAP = PROJECTMAP / "resultaten"

print(f"Invoer: {CSV_PAD}")
print(f"Uitvoer: {UITVOERMAP}")

## 3. Controleer eerst de structuur

Deze stap start het model nog niet. Hij controleert of de CSV kan worden ingelezen en laat alleen aantallen zien.

In [ ]:
data = laad_momants_csv(CSV_PAD)
bezoekersberichten = selecteer_bezoekersberichten(data)

print(f"Berichtrijen: {len(data)}")
print(f"Bruikbare bezoekersberichten: {len(bezoekersberichten)}")
print(f"Gesprekken: {bezoekersberichten['conversation_id'].nunique()}")

## 4. Extraheer het sentiment

Deze stap laadt het TabularisAI-model en schrijft `sentiment_per_bericht.csv` en `sentiment_per_gesprek.csv`. Het gesprekssentiment volgt het chronologisch laatste bezoekersbericht.

In [ ]:
berichtresultaten, gesprekresultaten = verwerk_csv(
    csv_pad=CSV_PAD,
    uitvoermap=UITVOERMAP,
    batchgrootte=32,
    tekst_opnemen=False,
)

gesprekresultaten.head(10)